# LLM · 实验工作台
按 block 依次运行：**配置 → ChArUco 标定 / 复用验证 → AprilTag 装置定位 → 试录 → 选 pattern → 正式实验 → 后处理**。

Mac / Windows 共用；在 VS Code 选择本项目 `.venv` kernel（新电脑先 `uv sync --python 3.13`）。

- 刚性 holder 整体移动：内参和双目相对位姿在理想情况下不变；先在最终机位拍新板子验证，PASS 才复用。变焦/对焦、单台相机移动、支架变形、裁剪模式改变：重新标定。
- 即使双目标定复用，相机相对装置的位置已经变了：重新拍 AprilTags + 垂线，建立 bearing 原点与重力坐标。
- 当前用户已刷 **flight**：reset 回到 IDLE，不会自行跑 ramp。正式 block 自动 reset/检查 READY；之后点 Record，再点 START pattern + LED。**录制中不再按硬件 reset**。
- 本 notebook 的正式实验是 frequency pattern 开环记录，不启用 hover/姿态闭环。原始 CS 仍是低频滤波电流估计，不能恢复真实 current phase。
- 所有运行按钮都需要你实际点击；Run All 不会开始驱动，但会打开采集界面。Stop / finish 会先停止驱动再完成视频；GPIO14 是已有硬件停止入口。不要强杀 kernel。

已有 notebook（含历史输出）备份在 `LLM/runs/notebook_backups/`。

**本次只验证 pattern / LED / CS：Block 0 设 `MODE="sync_check"`，依次运行 0 → 1 → 4 → 6 → 7 → 8 → 9，跳过 2、3、5。** 白线先保存为参考照片；不会声称已经定义三维 z 或测到轨迹。需完整三维实验时改为 `MODE="measured"`，按全部 block 完成验证。不要 Run All。


## Block 0 · 环境与本次 session
改了配置后重跑本 block，所有验证状态会清空。后续只使用 SESSION 里的冻结配置。A/B 需用实物标签核对，不以 USB index 作为永久身份。

In [ ]:
from pathlib import Path
import sys, json
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p/'LLM').is_dir() and (p/'controller/camera').is_dir()), None)
if PROJECT is None:
    raise RuntimeError('请先把 notebook 放在 ESP32_PMW/LLM，并从项目启动 kernel')
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
from IPython.display import display, Image
from LLM.io import read_json, write_json
from LLM.experiment import new_session, audit_take, prepare_run, prepare_sync_run, finish_take, postprocess
from LLM.experiment_panel import run_panel
from LLM.setup_geometry import verify_bag, localize_fixture, inspect_tags
from controller.camera import session as camera_session

MODE = "sync_check"  # sync_check: 同步联调；measured: 已标定三维实验
if MODE not in ("sync_check", "measured"): raise ValueError("Unknown MODE")

CAMERA = dict(indices=(1, 0), width=1280, height=800, fps=120,
              rotate180=True, backend='auto', codec='auto')
SESSION = new_session(PROJECT/'LLM/runs/sessions', CAMERA)
SESSION_DIR = Path(SESSION['directory'])
RIG = PROJECT/'LLM/runs/setup_002/calibration/stereo_rig.json'
# Windows 上请先复制对应 rig 文件；不要拿旧 640x400 标定替代。
VERIFICATION = WORLD = TEST_REPORT = RUN = LAST_RESULT = REFERENCE_PAIR = None
print('Session:', SESSION_DIR, '\n冻结相机配置:', SESSION['camera'])
display(camera_session.check_environment())


## Block 1 · 找到相机，核对 A/B
关闭其他占用相机的程序。下方 browser panel 打开后，遮挡一个镜头确认 A/B，调好视野、对焦；标定开始后不再调镜头。点 Close preview 返回 notebook。可选 probe 会逐台打开相机。

In [ ]:
PROBE = False
if PROBE:
    display(camera_session.probe(max_index=6, width=CAMERA['width'], height=CAMERA['height'],
                                fps=CAMERA['fps'], backend=CAMERA['backend']))
preview = run_panel(SESSION, output_root=SESSION_DIR/'preview_takes', max_seconds=10)


## Block 2 · 完整 ChArUco 标定（首次或验证失败时）
默认先尝试复用昨天通过的 setup_002：此 block 可跳过。新标定设 `DO_NEW_CALIBRATION=True`。

板子：9×6 squares，DICT_4X4_100，square 16.52 mm，marker 12.39 mm。后者按比例推算，尚非独立量测。
把板子放不同位置、深度，向不同方向倾斜；每处停稳。保存约 20–30 有效 pairs，并让每台都有足够单目覆盖；数量不替代验证。
运行中自动采集；notebook 按 **Interrupt 一次** 结束采集后会继续求解。不要 Restart。
每次用新目录，不跨 holder 搬动 append。失败不会替换当前 RIG。

In [ ]:
DO_NEW_CALIBRATION = False
if DO_NEW_CALIBRATION:
    from uuid import uuid4
    from controller.calib.calibrate import main as calibrate
    VERIFICATION = WORLD = None
    folder = SESSION_DIR/('calibration_'+uuid4().hex[:8])
    candidate = folder/'stereo_rig.json'
    args = ['--bag', str(folder/'images'), '--rig', str(candidate), '--out', str(folder/'diagnostics'),
            '--indices', *map(str, SESSION['camera']['indices']), '--backend', SESSION['camera']['backend'],
            '--board', '9x6_letter', '--square-mm', '16.52', '--marker-mm', '12.39']
    if (SESSION['camera']['width'],SESSION['camera']['height']) != (1280,800):
        raise ValueError('当前完整标定入口采集 1280x800；先把 session 改成此模式')
    if not SESSION['camera']['rotate180']: args += ['--no-flip']
    if calibrate(args) != 0: raise RuntimeError('新标定 FAIL；查看 diagnostics，不放宽门限')
    RIG = candidate
print('待验证的标定:', RIG)


## Block 3 · 在最终机位验证旧/新标定
先将 holder 放到实验位置，然后采一小组 **新的** ChArUco 图片：至少 5 个不同位置/倾角的静止 pairs。用 Interrupt 一次结束。
验证只求板子的姿态，K、畸变、双目外参全部固定；不会通过重新拟合相机来掩盖移动。
门限是工程验收起点：每组 RMS <1 px、最差点 <3 px，法向跨度 ≥10°、位置跨度 ≥20 mm。PASS 不等于已经验证毫米精度。
验证失败：先看 A/B、旋转/尺寸、运动模糊；必要时回 Block 2 重新标定。

In [ ]:
from uuid import uuid4
from controller.calib.calibrate import CharucoSpec
from controller.calib.capture import capture
VERIFICATION = WORLD = None
bag = SESSION_DIR/('verify_'+uuid4().hex[:8])/'images'
c = SESSION['camera']
spec = CharucoSpec(9,6,16.52,12.39,'DICT_4X4_100','9x6_letter')
capture(bag, indices=c['indices'], spec=spec, width=c['width'], height=c['height'],
        rotate180=c['rotate180'], backend=c['backend'])
VERIFICATION = verify_bag(bag, RIG, c, session_id=SESSION['session_id'])
display(VERIFICATION)
if not VERIFICATION['passed']: raise RuntimeError('验证未通过；不能进入已标定的正式实验')


## Block 4 · 装置 AprilTags：采集与检查
移走 ChArUco 板，保持相机和装置不动。让静止垂线的两个对应物理点在 **两台** 相机可见；单台图像里的线方向不能独立确定三维重力方向。

点 Save static reference pair，然后 Close preview。程序保存无压缩 PNG 和真实时间，自动标出 tag36h11 ID。
四个角块×三个面=12 张，但这还不足以推出 ID 到 CAD 面的对应；在 `LLM/config/fixture_layout.json` 填 ID 的四角 CAD 坐标、bearing 中心、固定 x 方向。25 mm 应是含黑边的编码区域外边长，不含纸张白边。
每张 tag 四角顺序是该编码标准方向的左上、右上、右下、左下；不是图片画面里的左上等。
至少两张已知 tag 同时出现在两台相机中。

**sync_check：只需 Save static reference pair → Close preview，保存白线和 LED 的原始图即可，不要求识别 tag。下面显示的像素坐标轴用于填写 Block 7 的 LED ROI。白线两端须是两机可辨认的相同物理点；本轮只留证据，不自动重建 z。**


In [ ]:
WORLD = None
reference_capture = run_panel(SESSION, output_root=SESSION_DIR/'reference_takes', max_seconds=10)
REFERENCE_PAIR = reference_capture['reference_pair']
if REFERENCE_PAIR is None: raise ValueError('请保存一对 reference，再关闭 preview')
if MODE == 'measured':
    display(inspect_tags(REFERENCE_PAIR))
    for tag in 'AB': display(Image(filename=str(Path(REFERENCE_PAIR)/f'{tag}_tag_ids.png'), width=800))
else:
    import matplotlib.pyplot as plt
    import cv2
    for tag in 'AB':
        frame = cv2.imread(str(Path(REFERENCE_PAIR)/f'{tag}.png'))
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set(title=f'Camera {tag}: original pixel coordinates', xlabel='x (pixels)', ylabel='y (pixels)')
        ax.grid(alpha=.25)
        display(fig)
        plt.close(fig)
    print('参考图已保存。sync_check 跳过 Block 5，继续 Block 6。')


## Block 5 · 装置定位 + 重力坐标验证
第一次补齐 CAD 对应表后，将 `survey_confirmed` 改成 true。`PLUMB` 填本次 reference 图中静止垂线两端的对应像素（原始 1280×800 图，非缩放预览坐标）。上下点必须在两张图里是同一个物理点。
程序验证 tag 重投影、垂线双目一致性，再计算 origin=bearing，z=垂线向上，x=装置固定 x 方向在垂直 z 平面上的投影，y=z×x。
因此相机整体移动后仍可恢复相同的实验 XY，而不是每次跟着相机右侧转。

In [ ]:
LAYOUT = PROJECT/'LLM/config/fixture_layout.json'
PLUMB = None
# PLUMB = {'bottom': {'a':[xA,yA], 'b':[xB,yB]},
#          'top':    {'a':[xA,yA], 'b':[xB,yB]}}
WORLD = None
if PLUMB is None or not read_json(LAYOUT)['survey_confirmed']:
    print('待补齐 CAD ID/角点/bearing 与本次垂线像素。可先做下面的相机试录。')
else:
    WORLD = localize_fixture(REFERENCE_PAIR, RIG, LAYOUT, PLUMB, SESSION['camera'],
                             session_id=SESSION['session_id'])
    write_json(SESSION_DIR/'world_frame.json', WORLD)
    display(WORLD)


## Block 6 · 10 秒试录与文件验证
点 Record 后自动录 10 秒；只用相机，不连串口。结束后自动逐帧解码验收。修改相机模式或新 session 后必须重做。

In [ ]:
TEST_REPORT = None
trial = run_panel(SESSION, output_root=SESSION_DIR/'trial_takes', max_seconds=10)
if trial['take'] is None: raise ValueError('未录制；请点 Record')
TEST_REPORT = audit_take(trial['take'], SESSION)
display(TEST_REPORT)


## Block 7 · 选择 pattern 与实验条件
pattern 库在 `LLM/patterns/`，复制 JSON 后修改，命名示例 `rise_2to210Hz_30s_ease_k2`、`rise_then_fall_...`。下拉选择后运行下一格预览。
现有文件是历史 ramp.DEFAULT 的明确副本，不代表当前 robot 已通过实验验证。每次运行都会把选定 JSON 原样快照保存；实际发送也由该快照生成。
当前自动执行支持公共 frequency，不执行四路 channel_updates；phase/duty 未配置时明确留空。

**sync_check 同样需要填实际端口、LED ROI 与实验说明。ROI 是原图上的 `[左上角x, 左上角y, 宽, 高]`，不是缩小预览中的坐标；紧贴发光区域，避免大块黑背景稀释亮度。**


In [ ]:
import ipywidgets as widgets
PATTERN_FILES = sorted((PROJECT/'LLM/patterns').glob('*.json'))
pattern_picker = widgets.Dropdown(options=[(p.stem,str(p)) for p in PATTERN_FILES], description='Pattern:')
display(pattern_picker)


In [ ]:
from LLM.pattern import sample_pattern
SELECTED_PATTERN = Path(pattern_picker.value)
plan = sample_pattern(read_json(SELECTED_PATTERN))
display(plan.plot(x='t_s', y='frequency_hz', grid=True, title=SELECTED_PATTERN.stem))

from serial.tools import list_ports
print('可选串口（这里只列出，不连接）:', [p.device for p in list_ports.comports()])

SERIAL_PORT = "/dev/cu.usbserial-0001"       # 本机 ESP 串口，例如 /dev/cu... 或 COM...；不要猜
ROI_A = [830, 500, 32, 36]
ROI_B = [734, 474, 32, 32]   # 各机 LED 的原始图像 [x,y,width,height]
NOTE = "电源debug"              # robot / bearing / coil setup、实验目的
WARNING_REVIEW = (
    "已检查试录：1208 对中 1 对 host 时间差超过 10 ms，最大 10.48 ms；"
    "文件完整性 PASS，编码队列无丢帧。"
    "本次仅进行 sync_check，后续分别检测两路 LED 边沿，"
    "不将 host 时间差视为曝光同步验证，不用于定量三维运动分析。"
)

## Block 7B · 可选：只发 frequency pattern，不录像
用于单独排查电源电流。先停止 Block 8/其他串口程序。只需 Block 0 提供 PROJECT，Block 7 选择 pattern 并填写 SERIAL_PORT；不需要相机、ROI、试录或标定。

将下面 RUN_DRIVE_ONLY 改为 True，然后单独运行此格：自动 reset / READY → 立即发送下拉框当前选中的 pattern → 显示设备频率与四路 CS → 到时 stop。当前 flight 仍会点亮 LED；没有录像或光学同步验证。日志独立保存在 LLM/runs/drive_only/，不能交给 Block 9 当作视频实验。

中途停止用 notebook 的 Interrupt；D14/供电停止仍可用。USB 断开或进程强杀不能保证软件停止。不要 Run All；测试结束后将代码中的开关改回 False。

Block 7B/8 已取消模型冷却等待，由你判断线圈状态后运行。若旧 cell 还在等待，先 Interrupt 再运行更新后的代码；本格会重新加载入口。输出先显示实际 pattern 和驱动时长。


In [ ]:
import importlib
import controller.control.hover_controller_runner as _drive_runner
import LLM.drive_only as _drive_test

RUN_DRIVE_ONLY = False  # 要启动这次无录像测试时，改成 True 并单独运行本格
if RUN_DRIVE_ONLY:
    RUN_DRIVE_ONLY = False
    # Reload the updated helper without restarting the notebook kernel.
    importlib.reload(_drive_runner)
    importlib.reload(_drive_test)
    # Read the current dropdown choice, not a stale preview variable.
    DRIVE_ONLY_PATTERN = Path(pattern_picker.value)
    DRIVE_ONLY_RESULT = _drive_test.run_frequency_pattern(
        DRIVE_ONLY_PATTERN,
        SERIAL_PORT,
        output_root=PROJECT / 'LLM/runs/drive_only',
    )
    print('无录像测试日志:', DRIVE_ONLY_RESULT)
else:
    print('未启动。要测试线圈，将 RUN_DRIVE_ONLY 改为 True，单独运行本格。')


## Block 8 · 正式实验：Record → START → 自动 Stop → 保存
本格会打开相机和唯一串口，自动 reset flight 并等待 READY；初始保持 preview。

1. 浏览器核对 A/B、相机/装置未移动；LED 初始灭，两个 ROI 覆盖它。
2. 点 Record；至少录入 0.25 s 暗帧，START 按钮才可用。
3. 点 START pattern + LED。不用按 ESP reset。固件 START 时间与原始 CS 日志一起保存。
4. 到选定 pattern 的末端，发送 stop，再录 2 s；也可随时点 Stop / finish。录制总限时包含 30 s 操作余量，等待太久会禁用 START。
5. 返回后自动审核这次新录像并导入本次 run；不会沿用旧 REPORT。

程序运行在主线程；浏览器按钮只提交事件。没有 live pose/hover 控制。USB 断开/进程被强杀不保证停机，沿用 GPIO14/实验供电停止流程。
`MODE="sync_check"` 不要求 CAD/bearing 或 rig 验证；仍检查同 session 的试录和参考照片。`MODE="measured"` 保留所有三维实验前置要求。

运行前填好 Block 7；Record 后建议等约 1 秒再点 START。这一格会操作线圈，不要 Run All。

In [ ]:
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, "-m", "LLM.serial_check",
        "--port", "/dev/cu.usbserial-0001",
    ],
    cwd=PROJECT,
)

In [ ]:
from controller.control import ramp
LAST_RESULT = None
if not SERIAL_PORT or ROI_A is None or ROI_B is None:
    raise ValueError('先在 Block 7 填实际 SERIAL_PORT、ROI_A、ROI_B')
RUN = None
if MODE == 'sync_check':
    RUN = prepare_sync_run(PROJECT/'LLM/runs', SESSION, TEST_REPORT or {}, REFERENCE_PAIR,
                           note=NOTE, warning_review=WARNING_REVIEW)
elif MODE == 'measured':
    RUN = prepare_run(PROJECT/'LLM/runs', SESSION, RIG, VERIFICATION or {}, WORLD or {},
                      TEST_REPORT or {}, note=NOTE, warning_review=WARNING_REVIEW)
else:
    raise ValueError('回到 Block 0 选择 MODE')
# 选中的文件在打开硬件前冻结；该 block 每次运行创建新的 run。
from LLM.io import snapshot
snapshot(SELECTED_PATTERN, RUN/'pattern.json')
selected = read_json(RUN/'pattern.json')
# 重新加载入口，无需重置当前 session。
import importlib
import LLM.experiment_panel as _experiment_panel
run_panel = importlib.reload(_experiment_panel).run_panel
LAST_RESULT = run_panel(SESSION, output_root=RUN/'raw/captures', run=RUN,
                        pattern_path=RUN/'pattern.json', serial_port=SERIAL_PORT,
                        roi_a=ROI_A, roi_b=ROI_B,
                        max_seconds=ramp.duration_s(selected['frequency_segments'])+30)
if LAST_RESULT['take'] is None: raise ValueError('只看了 preview，未产生录像；正式重拍需新 run')
report = finish_take(LAST_RESULT['take'], SESSION, RUN)
display(report)
print('实验包:', RUN)


## Block 9 · 后处理：LED 对时 → 轨迹 → 起飞 → 曲线 / 共同窗口
先自动对齐两路 LED 与唯一设备 START，再做轨迹。默认需要同一物理点的双目标注 CSV；师兄的 detector 也可输出该格式。
只有你明确选择 `EXPERIMENTAL_AUTO=True` 才试现有 ellipse center，结果只是未经实物验证的位置代理，轴角无效。
缺少数据会停在对应步骤，不生成假的三幅图。takeoff baseline/阈值在本次 run 的 config.json 中调整；不要把标定残差当作实物精度。
CS 曲线仍是约 2 Hz 的滤波估计；真实电流幅值和 phase 需要另外的采样链路。
交付整个 run，包含原视频、时间戳、pattern、serial、标定/装置参考与导出。

**sync_check：保留 `ANNOTATIONS=None` 和 `EXPERIMENTAL_AUTO=False`。输出 frequency、四路低频 CS、两机 LED 亮度三个面板，状态为 `sync_check_inputs_only`；不求轨迹、起飞或三维 z。LED 曲线按各自边沿对齐，所以重合不证明曝光同步。`INCOMPLETE.txt` 指完整三维数据包尚未完成，不否定本次输入诊断输出。**


In [ ]:
ANNOTATIONS = None       # 真实物理点 CSV 路径；见 SCHEMA.md
EXPERIMENTAL_AUTO = False
if RUN is None: raise ValueError('先完成并导入一次正式实验')
result = postprocess(RUN, ROI_A, ROI_B, annotations=ANNOTATIONS, experimental_auto=EXPERIMENTAL_AUTO)
display(result)
if (RUN/'export/input_curves.png').exists():
    display(Image(filename=str(RUN/'export/input_curves.png')))
if (RUN/'export/overview.png').exists() and not (RUN/'export/INCOMPLETE.txt').exists():
    display(Image(filename=str(RUN/'export/overview.png')))
